In [ ]:
import sys
import json
import numpy as np
import torch
import cv2
import trimesh
from pathlib import Path
from scipy.spatial.transform import Rotation
from PIL import Image
import nvdiffrast.torch as dr

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"   # use the less-busy GPU

sys.path.insert(0, str(Path.home() / "rai_workspace/FoundationPose"))

try:
    from Utils import erode_depth
    print("ok")
except Exception as e:
    print(type(e).__name__, ":", e)

ok


In [ ]:
PROJECT_ROOT = Path(".").resolve().parent

FP_ROOT   = Path.home() / "rai_workspace/FoundationPose"
SAM2_ROOT = Path.home() / "rai_workspace/sam2"
sys.path.insert(0, str(FP_ROOT))
sys.path.insert(0, str(FP_ROOT / "mycuda"))
sys.path.insert(0, str(SAM2_ROOT))

GD_CONFIG  = Path.home() / "rai_workspace/grounding_dino_weights/GroundingDINO_SwinT_OGC.py"
GD_WEIGHTS = Path.home() / "rai_workspace/grounding_dino_weights/groundingdino_swint_ogc.pth"
SAM2_CFG = "configs/sam2.1/sam2.1_hiera_l"
SAM2_CKPT  = Path.home() / "rai_workspace/sam2_weights/sam2.1_hiera_large.pt"
MESH_PATH  = PROJECT_ROOT / "megapose_objects/lamp/mesh.ply"
CAM_INFO   = PROJECT_ROOT / "outputs/camera_info.json"

DETECTION_PROMPT = "yellow lamp"
BOX_THRESHOLD    = 0.25
TEXT_THRESHOLD   = 0.20
MAX_PER_CAMERA   = 2      # keep up to 2 detections per camera
MAX_TOTAL        = 8      # hard cap on FoundationPose runs
CLUSTER_DIST_M   = 0.30   # world-frame radius for merging duplicate estimates
N_FP_ITERATIONS  = 5      # FoundationPose register() refinement iterations
DEVICE           = "cuda"

# Ground-truth poses [x, y, z, qw, qx, qy, qz] from mesh_clutter_high_30.g
GT = {
    "goal_yellow_lamp_9_mesh":       np.array([-0.620966, -0.342511,  0.671156,
                                                0.287543, -0.46152,   0.714029,  0.441]),
    "goal_pose_yellow_lamp_12_mesh": np.array([ 0.219547,  0.280159,  0.698313,
                                                0.38692,   0.0,       0.0,      -0.922113]),
}

In [4]:
def boxes_cxcywh_to_xyxy(boxes_norm: torch.Tensor, W: int, H: int) -> torch.Tensor:
    cx, cy, w, h = boxes_norm.unbind(-1)
    return torch.stack([(cx - w/2)*W, (cy - h/2)*H,
                        (cx + w/2)*W, (cy + h/2)*H], dim=-1)


def project_pt(pt3d, K):
    p = K @ np.asarray(pt3d, dtype=float)
    return (int(p[0]/p[2]), int(p[1]/p[2]))

In [5]:
print("Loading GroundingDINO …")
from groundingdino.util.inference import load_model, load_image, predict
gd_model = load_model(str(GD_CONFIG), str(GD_WEIGHTS))
gd_model.eval()

Loading GroundingDINO …


final text_encoder_type: bert-base-uncased


[_send_single_request()] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/config.json "HTTP/1.1 200 OK"
[_send_single_request()] HTTP Request: HEAD https://huggingface.co/bert-base-uncased/resolve/main/tokenizer_config.json "HTTP/1.1 200 OK"
[_send_single_request()] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 307 Temporary Redirect"
[_send_single_request()] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"
[_send_single_request()] HTTP Request: GET https://huggingface.co/api/models/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 307 Temporary Redirect"
[_send_single_request()] HTTP Request: GET https://huggingface.co/api/models/google-bert/bert-base-uncased/tree/main?recursive=true&expand=false "HTTP/1.1 200 OK"
[_send_single

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GroundingDINO(
  (transformer): Transformer(
    (encoder): TransformerEncoder(
      (layers): ModuleList(
        (0-5): 6 x DeformableTransformerEncoderLayer(
          (self_attn): MultiScaleDeformableAttention(
            (sampling_offsets): Linear(in_features=256, out_features=256, bias=True)
            (attention_weights): Linear(in_features=256, out_features=128, bias=True)
            (value_proj): Linear(in_features=256, out_features=256, bias=True)
            (output_proj): Linear(in_features=256, out_features=256, bias=True)
          )
          (dropout1): Dropout(p=0.0, inplace=False)
          (norm1): LayerNorm((256,), eps=1e-05, elementwise_affine=True)
          (linear1): Linear(in_features=256, out_features=2048, bias=True)
          (dropout2): Dropout(p=0.0, inplace=False)
          (linear2): Linear(in_features=2048, out_features=256, bias=True)
          (dropout3): Dropout(p=0.0, inplace=False)
          (norm2): LayerNorm((256,), eps=1e-05, elementwise_aff

In [6]:
print("Loading SAM2 …")
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

sam2_model     = build_sam2(SAM2_CFG, str(SAM2_CKPT), device=DEVICE)
sam2_predictor = SAM2ImagePredictor(sam2_model)

Loading SAM2 …


[_load_checkpoint()] Loaded checkpoint sucessfully


In [7]:
print("Loading FoundationPose …")
from estimater import FoundationPose, PoseRefinePredictor, ScorePredictor

scorer  = ScorePredictor()
refiner = PoseRefinePredictor()
glctx   = dr.RasterizeCudaContext()

mesh = trimesh.load(str(MESH_PATH))
fp_estimator = FoundationPose(
    model_pts     = np.array(mesh.vertices,       dtype=np.float32),
    model_normals = np.array(mesh.vertex_normals,  dtype=np.float32),
    mesh          = mesh,
    scorer        = scorer,
    refiner       = refiner,
    debug_dir     = "/tmp/fp_debug",
    debug         = 0,
    glctx         = glctx,
)

[__init__()] self.cfg: 
 lr: 0.0001
c_in: 6
zfar: 'Infinity'
debug: null
n_view: 1
run_id: 3wy8qqex
use_BN: true
exp_name: 2024-01-11-20-02-45
n_epochs: 62
save_dir: /home/bowenw/debug/2024-01-11-20-02-45/
use_mask: false
loss_type: pairwise_valid
optimizer: adam
batch_size: 64
crop_ratio: 1.1
enable_amp: true
use_normal: false
max_num_key: null
warmup_step: -1
input_resize:
- 160
- 160
max_step_val: 1000
vis_interval: 1000
weight_decay: 0
normalize_xyz: true
resume_run_id: null
clip_grad_norm: 'Infinity'
lr_epoch_decay: 500
render_backend: nvdiffrast
train_num_pair: 5
lr_decay_epochs:
- 50
n_epochs_warmup: 1
make_pair_online: false
gradient_max_norm: 'Infinity'
max_step_per_epoch: 10000
n_rendering_workers: 1
save_epoch_interval: 100
n_dataloader_workers: 100
split_objects_across_gpus: true
ckpt_dir: /home/salman/rai_workspace/FoundationPose/learning/training/../../weights/2024-01-11-20-02-45/model_best.pth

[__init__()] self.h5_file:None
[__init__()] Using pretrained model from /home

Loading FoundationPose …


[__init__()] Using pretrained model from /home/salman/rai_workspace/FoundationPose/learning/training/../../weights/2023-10-28-18-33-37/model_best.pth
[__init__()] init done
[reset_object()] self.diameter:0.09867342099723961, vox_size:0.004933671049861981
[reset_object()] self.pts:torch.Size([162, 3])
[reset_object()] reset done
[make_rotation_grid()] cam_in_obs:(42, 4, 4)
[make_rotation_grid()] rot_grid:(252, 4, 4)
[make_rotation_grid()] after cluster, rot_grid:(252, 4, 4)
[make_rotation_grid()] self.rot_grid: torch.Size([252, 4, 4])


num original candidates = 252
num of pose after clustering: 252


In [8]:
with open(CAM_INFO) as f:
    camera_info = json.load(f)

print(f"\nDetecting: '{DETECTION_PROMPT}'  (threshold={BOX_THRESHOLD})")
all_detections: list[dict] = []

for cam_name, info in camera_info.items():
    W, H = info["width"], info["height"]
    _, img_tensor = load_image(info["rgb_path"])

    boxes_norm, logits, phrases = predict(
        model          = gd_model,
        image          = img_tensor,
        caption        = DETECTION_PROMPT,
        box_threshold  = BOX_THRESHOLD,
        text_threshold = TEXT_THRESHOLD,
    )

    if len(logits) == 0:
        print(f"  {cam_name}: no detection")
        continue

    boxes_xyxy = boxes_cxcywh_to_xyxy(boxes_norm, W, H)
    order      = logits.argsort(descending=True)[:MAX_PER_CAMERA]

    # Load once per camera for SAM2
    rgb_np = np.array(Image.open(info["rgb_path"]).convert("RGB"))
    sam2_predictor.set_image(rgb_np)

    for i in order:
        score = logits[i].item()
        box   = boxes_xyxy[i].numpy()
        print(f"  {cam_name}: score={score:.3f}  box={box.astype(int).tolist()}")

        masks, _, _ = sam2_predictor.predict(
            point_coords     = None,
            point_labels     = None,
            box              = box[None],   # SAM2 expects (1,4) xyxy
            multimask_output = False,
        )
        mask = masks[0].astype(bool)        # (H, W) boolean

        if not mask.any():
            print(f"    SAM2 returned empty mask — skipping")
            continue

        all_detections.append({
            "score":    score,
            "cam":      cam_name,
            "box_xyxy": box,
            "mask":     mask,
            "info":     info,
        })

if not all_detections:
    raise RuntimeError(
        "No detections found. Lower BOX_THRESHOLD or re-run step2 to regenerate images."
    )

all_detections.sort(key=lambda d: d["score"], reverse=True)
top_dets = all_detections[:MAX_TOTAL]
print(f"\n{len(top_dets)} detection(s) selected for FoundationPose")


Detecting: 'yellow lamp'  (threshold=0.25)


[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!
[set_image()] For numpy array image, we assume (HxWxC) format
[set_image()] Computing image embeddings for the provided image...
[set_image()] Image embeddings computed.


  cam_dim_0: score=0.539  box=[255, 1280, 331, 1391]
  cam_dim_0: score=0.448  box=[731, 1696, 886, 1825]


[set_image()] For numpy array image, we assume (HxWxC) format
[set_image()] Computing image embeddings for the provided image...
[set_image()] Image embeddings computed.


  cam_dim_1: score=0.510  box=[96, 405, 1824, 769]
  cam_dim_1: score=0.351  box=[1014, 377, 1128, 434]


...

  cam_dim_2: score=0.532  box=[97, 423, 1823, 767]
  cam_dim_2: score=0.337  box=[253, 429, 805, 596]


[set_image()] For numpy array image, we assume (HxWxC) format
[set_image()] Computing image embeddings for the provided image...
[set_image()] Image embeddings computed.


  cam_dim_3: score=0.455  box=[96, 400, 1824, 768]
  cam_dim_3: score=0.377  box=[240, 443, 336, 521]


[set_image()] For numpy array image, we assume (HxWxC) format
[set_image()] Computing image embeddings for the provided image...
[set_image()] Image embeddings computed.


  cam_dim_4: score=0.452  box=[96, 425, 1823, 767]
  cam_dim_4: score=0.371  box=[638, 556, 892, 678]

8 detection(s) selected for FoundationPose


In [ ]:
sam2_model.to("cpu")
torch.cuda.empty_cache()

print("\nRunning FoundationPose …")
estimates: list[dict] = []

for idx, det in enumerate(top_dets):
    info = det["info"]
    print(f"  [{idx+1}/{len(top_dets)}]  cam={det['cam']}  score={det['score']:.3f}")

    rgb   = np.array(Image.open(info["rgb_path"]).convert("RGB"))
    depth = np.load(info["depth_path"]).astype(np.float32)
    K     = np.array(info["K"],           dtype=np.float64)
    T_wc  = np.array(info["T_world_cam"], dtype=np.float64)
    mask  = det["mask"]

    try:
        T_co = fp_estimator.register(
            K         = K,
            rgb       = rgb,
            depth     = depth,
            ob_mask   = mask,
            iteration = N_FP_ITERATIONS,
        )
    except Exception as e:
        print(f"    FoundationPose failed: {e}")
        continue

    T_wo      = T_wc @ T_co
    pos       = T_wo[:3, 3]
    quat_xyzw = Rotation.from_matrix(T_wo[:3, :3]).as_quat()
    quat_wxyz = np.array([quat_xyzw[3], *quat_xyzw[:3]])   # RAI convention

    print(f"    → world pos = {np.round(pos, 4)}")
    estimates.append({
        "position":    pos,
        "quat_wxyz":   quat_wxyz,
        "T_world_obj": T_wo,
        "T_cam_obj":   T_co,
        "score":       det["score"],
        "cam":         det["cam"],
        "box_xyxy":    det["box_xyxy"],
        "mask":        mask,
        "info":        info,
    })

if not estimates:
    raise RuntimeError("FoundationPose produced no estimates. Check GPU memory and logs above.")

print(f"\n{len(estimates)} pose estimate(s) obtained")

[register()] Welcome
[register()] poses:(252, 4, 4)
[register()] after viewpoint, add_errs min:-1.0



Running FoundationPose …
  [1/8]  cam=cam_dim_0  score=0.539
Module Utils 9293bed load on device 'cuda:0' took 0.32 ms  (cached)


[predict()] ob_in_cams:(252, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()

    → world pos = [-0.621  -0.3421  0.6704]
  [2/8]  cam=cam_dim_2  score=0.532


[predict()] ob_in_cams:(252, 4, 4)
[predict()] self.cfg.use_normal:False
[predict()] trans_normalizer:[0.019999999552965164, 0.019999999552965164, 0.05000000074505806], rot_normalizer:0.3490658503988659
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()

...

    → world pos = [-0.1721 -0.7722  0.6767]
  [7/8]  cam=cam_dim_3  score=0.377


[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_cr

    → world pos = [-0.1147 -0.7301  0.6672]
  [8/8]  cam=cam_dim_4  score=0.371


[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_crop_data_batch()] make tf_to_crops done
[make_crop_data_batch()] render done
[make_crop_data_batch()] warp done
[make_crop_data_batch()] pose batch data done
[predict()] forward start
[predict()] forward done
[predict()] making cropped data
[make_crop_data_batch()] Welcome make_crop_data_batch
[make_cr

    → world pos = [-0.6321  0.1548  0.6694]

8 pose estimate(s) obtained


In [10]:
clusters: list[dict] = []
for est in estimates:
    placed = False
    for cl in clusters:
        if np.linalg.norm(est["position"] - cl["rep"]["position"]) < CLUSTER_DIST_M:
            cl["members"].append(est)
            if est["score"] > cl["rep"]["score"]:
                cl["rep"] = est
            placed = True
            break
    if not placed:
        clusters.append({"rep": est, "members": [est]})

clusters.sort(key=lambda c: c["rep"]["score"], reverse=True)
print(f"{len(clusters)} unique object(s) found after clustering\n")

6 unique object(s) found after clustering



In [11]:
print("=" * 70)
print("POSE ESTIMATION RESULTS  (FoundationPose)")
print("=" * 70)

for i, cl in enumerate(clusters):
    rep = cl["rep"]
    pos = rep["position"]

    nearest_name = min(GT, key=lambda n: np.linalg.norm(pos - GT[n][:3]))
    pos_err      = np.linalg.norm(pos - GT[nearest_name][:3])

    print(f"\nObject {i+1}  →  likely '{nearest_name}'")
    print(f"  Estimated position   [x,y,z]   : {np.round(pos, 4)}")
    print(f"  Estimated quaternion [w,x,y,z] : {np.round(rep['quat_wxyz'], 4)}")
    print(f"  Best detection score           : {rep['score']:.3f}")
    print(f"  Best camera                    : {rep['cam']}")
    print(f"  Observations merged            : {len(cl['members'])}")
    print(f"  Position error vs GT           : {pos_err*100:.1f} cm")
    print(f"  4×4 T_world_obj:\n{rep['T_world_obj']}")

print("\n" + "=" * 70)
print("GROUND TRUTH (from mesh_clutter_high_30.g)")
print("=" * 70)
for name, gt in GT.items():
    print(f"  {name}:")
    print(f"    position   [x,y,z]   = {gt[:3]}")
    print(f"    quaternion [w,x,y,z] = {gt[3:]}")

POSE ESTIMATION RESULTS  (FoundationPose)

Object 1  →  likely 'goal_yellow_lamp_9_mesh'
  Estimated position   [x,y,z]   : [-0.621  -0.3421  0.6704]
  Estimated quaternion [w,x,y,z] : [ 0.2338 -0.3776  0.7606  0.4735]
  Best detection score           : 0.539
  Best camera                    : cam_dim_0
  Observations merged            : 1
  Position error vs GT           : 0.1 cm
  4×4 T_world_obj:
[[-0.605501   -0.79584223 -0.00197688 -0.62095898]
 [-0.35305771  0.26638931  0.89687628 -0.34213066]
 [-0.71324527  0.54375738 -0.44227719  0.67038965]
 [ 0.          0.          0.          1.        ]]

Object 2  →  likely 'goal_yellow_lamp_9_mesh'
  Estimated position   [x,y,z]   : [-0.0301 -0.4538  0.6659]
  Estimated quaternion [w,x,y,z] : [0.1412 0.1381 0.8295 0.5224]
  Best detection score           : 0.532
  Best camera                    : cam_dim_2
  Observations merged            : 2
  Position error vs GT           : 60.1 cm
  4×4 T_world_obj:
[[-0.92198789  0.08151667  0.37854

In [12]:
print("\nGenerating visualisations …")
mesh_obj  = trimesh.load(str(MESH_PATH))
verts     = np.array(mesh_obj.vertices)
verts_hom = np.hstack([verts, np.ones((len(verts), 1))])

PALETTE = [
    (0,   255,   0),    # green   — object 1
    (0,   165, 255),    # orange  — object 2
    (255,   0, 255),    # magenta
    (255, 255,   0),    # cyan
]

for i, cl in enumerate(clusters):
    rep      = cl["rep"]
    color    = PALETTE[i % len(PALETTE)]
    vis      = cv2.imread(rep["info"]["rgb_path"])
    H_v, W_v = vis.shape[:2]
    K_v      = np.array(rep["info"]["K"])
    T_co     = rep["T_cam_obj"]

    # semi-transparent mask overlay
    mask_u8  = rep["mask"].astype(np.uint8) * 255
    colored  = np.zeros_like(vis)
    colored[:] = color[::-1]   # BGR
    vis = cv2.addWeighted(
        vis, 1.0,
        cv2.bitwise_and(colored, colored, mask=mask_u8), 0.4,
        0
    )

    # bounding box
    x1, y1, x2, y2 = rep["box_xyxy"].astype(int)
    cv2.rectangle(vis, (x1, y1), (x2, y2), color, 4)

    # label
    nearest = min(GT, key=lambda n: np.linalg.norm(rep["position"] - GT[n][:3]))
    short   = "lamp" if "9_mesh" in nearest else "target"
    cv2.putText(vis, f"{short}  s={rep['score']:.2f}",
                (x1, max(y1 - 15, 30)), cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

    # XYZ axes: red=X, green=Y, blue=Z
    axis_len = 0.05
    o  = project_pt(T_co[:3, 3],                           K_v)
    xp = project_pt(T_co[:3, 3] + axis_len * T_co[:3, 0], K_v)
    yp = project_pt(T_co[:3, 3] + axis_len * T_co[:3, 1], K_v)
    zp = project_pt(T_co[:3, 3] + axis_len * T_co[:3, 2], K_v)
    cv2.arrowedLine(vis, o, xp, (0,   0, 255), 4, tipLength=0.3)
    cv2.arrowedLine(vis, o, yp, (0, 255,   0), 4, tipLength=0.3)
    cv2.arrowedLine(vis, o, zp, (255,   0,   0), 4, tipLength=0.3)

    # projected mesh vertices (every 8th for speed)
    verts_cam = (T_co @ verts_hom.T).T[:, :3]
    front = verts_cam[:, 2] > 0
    if front.any():
        pts  = (K_v @ verts_cam[front].T).T
        pts  = (pts[:, :2] / pts[:, 2:3]).astype(int)
        mask_pv = ((pts[:, 0] >= 0) & (pts[:, 0] < W_v) &
                   (pts[:, 1] >= 0) & (pts[:, 1] < H_v))
        for pt in pts[mask_pv][::8]:
            cv2.circle(vis, tuple(pt), 3, color, -1)

    # position text overlay
    pos = rep["position"]
    txt = f"pos=[{pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}]"
    cv2.putText(vis, txt, (20, H_v - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (255, 255, 255), 3)
    cv2.putText(vis, txt, (20, H_v - 30), cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 0, 0), 1)

    out_path = PROJECT_ROOT / "outputs" / f"pose_fp_object{i+1}_{short}.png"
    cv2.imwrite(str(out_path), vis)
    print(f"  Saved -> {out_path}")

print("\nDone.")


Generating visualisations …
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_object1_lamp.png
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_object2_lamp.png
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_object3_target.png
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_object4_target.png
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_object5_lamp.png
  Saved -> /home/salman/rai_workspace/foundation_pose_section/outputs/pose_fp_object6_lamp.png

Done.
